# Adventure Works — Retail Sales Optimization: ETL (Python)





In [1]:
%pip install pandas numpy sqlalchemy pymysql openpyxl jupyter ipykernel

Note: you may need to restart the kernel to use updated packages.


In [2]:
# libraries we need
%pip install pandas numpy sqlalchemy pymysql openpyxl jupyter ipykernel
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine


Note: you may need to restart the kernel to use updated packages.


## 1. Extract





In [3]:
# the dimension workbook has one sheet per table - read them all at once
dim_sheets = pd.read_excel("../Data/DimTables.xlsx", sheet_name=None)

dim_customer = dim_sheets["DimCustomer"]
dim_employee = dim_sheets["DimEmployee"]
dim_geography = dim_sheets["DimGeography"]
dim_product = dim_sheets["DimProduct"]
dim_reseller = dim_sheets["DimReseller"]
dim_sales_territory = dim_sheets["DimSalesTerritory"]

fact_internet_sales = pd.read_excel("../Data/FactInternetSales.xlsx")
fact_reseller_sales = pd.read_excel("../Data/FactResellerSales.xlsx")

tables = {
    "dim_customer": dim_customer,
    "dim_employee": dim_employee,
    "dim_geography": dim_geography,
    "dim_product": dim_product,
    "dim_reseller": dim_reseller,
    "dim_sales_territory": dim_sales_territory,
    "fact_internet_sales": fact_internet_sales,
    "fact_reseller_sales": fact_reseller_sales,
}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")


dim_customer: 18484 rows, 13 columns
dim_employee: 21 rows, 9 columns
dim_geography: 655 rows, 6 columns
dim_product: 397 rows, 7 columns
dim_reseller: 701 rows, 4 columns
dim_sales_territory: 11 rows, 4 columns
fact_internet_sales: 60398 rows, 13 columns
fact_reseller_sales: 60899 rows, 14 columns


## 2. Profile 




In [4]:
# dtypes and a first look at each table
for name, df in tables.items():
    print(f"\n--- {name} ---")
    print(df.dtypes)



--- dim_customer ---
CustomerKey                   int64
GeographyKey                  int64
CustomerName                    str
BirthDate            datetime64[us]
MaritalStatus                   str
Gender                          str
EmailAddress                    str
YearlyIncome                  int64
Education                       str
Occupation                      str
HouseOwnerFlag                int64
Address                         str
FirstPurchaseDate    datetime64[us]
dtype: object

--- dim_employee ---
EmployeeKey                   int64
ParentEmployeeKey           float64
SalesTerritoryKey             int64
EmployeeName                    str
Title                           str
EmailAddress                    str
DepartmentName                  str
HireDate             datetime64[us]
BirthDate            datetime64[us]
dtype: object

--- dim_geography ---
GeographyKey         int64
City                   str
State                  str
Country                str
Posta

In [5]:
# missing values per table
for name, df in tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(f"{name}: no missing values")
    else:
        print(f"{name}: missing values -")
        print(missing)


dim_customer: no missing values
dim_employee: missing values -
ParentEmployeeKey    1
dtype: int64
dim_geography: no missing values
dim_product: missing values -
Color    56
dtype: int64
dim_reseller: no missing values
dim_sales_territory: missing values -
SalesTerritoryRegion     1
SalesTerritoryCountry    1
SalesTerritoryGroup      1
dtype: int64
fact_internet_sales: missing values -
CarrierTrackingNumber    60398
dtype: int64
fact_reseller_sales: missing values -
ProductKey               44
ResellerKey              44
EmployeeKey              44
SalesTerritoryKey        44
SalesOrderNumber         44
SalesOrderLineNumber     44
DiscountAmount           44
TotalProductCost          3
SalesAmount               3
Freight                  44
CarrierTrackingNumber    44
OrderDate                44
DueDate                  44
ShipDate                 44
dtype: int64


In [6]:
# "NA" as literal text is a known issue in this dataset (e.g. some ProductSubcategoryKey
# and Color fields use the text "NA" instead of a real blank). That's not the same as a
# missing value until we tell pandas to treat it as one - check for it explicitly.
for name, df in tables.items():
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        na_count = (df[col] == "NA").sum()
        if na_count > 0:
            print(f"{name}.{col}: {na_count} literal 'NA' text values")


C:\Users\aviksomya.ext\AppData\Local\Temp\ipykernel_4828\4211767863.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns
C:\Users\aviksomya.ext\AppData\Local\Temp\ipykernel_4828\4211767863.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/do

In [7]:
# duplicate key checks
print("duplicate CustomerKey:", dim_customer["CustomerKey"].duplicated().sum())
print("duplicate ProductKey:", dim_product["ProductKey"].duplicated().sum())
print("duplicate EmployeeKey:", dim_employee["EmployeeKey"].duplicated().sum())
print("duplicate ResellerKey:", dim_reseller["ResellerKey"].duplicated().sum())
print("duplicate GeographyKey:", dim_geography["GeographyKey"].duplicated().sum())


duplicate CustomerKey: 0
duplicate ProductKey: 0
duplicate EmployeeKey: 0
duplicate ResellerKey: 0
duplicate GeographyKey: 0


In [8]:
# referential integrity - does every key used in the fact tables actually exist
# in the matching dimension table?
missing_customers = set(fact_internet_sales["CustomerKey"]) - set(dim_customer["CustomerKey"])
missing_products_internet = set(fact_internet_sales["ProductKey"]) - set(dim_product["ProductKey"])
missing_resellers = set(fact_reseller_sales["ResellerKey"]) - set(dim_reseller["ResellerKey"])
missing_products_reseller = set(fact_reseller_sales["ProductKey"]) - set(dim_product["ProductKey"])

print("CustomerKeys in fact_internet_sales with no match in dim_customer:", len(missing_customers))
print("ProductKeys in fact_internet_sales with no match in dim_product:", len(missing_products_internet))
print("ResellerKeys in fact_reseller_sales with no match in dim_reseller:", len(missing_resellers))
print("ProductKeys in fact_reseller_sales with no match in dim_product:", len(missing_products_reseller))


CustomerKeys in fact_internet_sales with no match in dim_customer: 0
ProductKeys in fact_internet_sales with no match in dim_product: 0
ResellerKeys in fact_reseller_sales with no match in dim_reseller: 44
ProductKeys in fact_reseller_sales with no match in dim_product: 44


In [9]:
# range / sanity checks on the money columns - nothing negative that shouldn't be
for name, df in [("fact_internet_sales", fact_internet_sales), ("fact_reseller_sales", fact_reseller_sales)]:
    print(f"\n{name}:")
    print("  negative SalesAmount:", (df["SalesAmount"] < 0).sum())
    print("  negative TotalProductCost:", (df["TotalProductCost"] < 0).sum())
    print("  negative Freight:", (df["Freight"] < 0).sum())
    print("  ShipDate before OrderDate:", (pd.to_datetime(df["ShipDate"]) < pd.to_datetime(df["OrderDate"])).sum())



fact_internet_sales:
  negative SalesAmount: 0
  negative TotalProductCost: 0
  negative Freight: 0
  ShipDate before OrderDate: 0

fact_reseller_sales:
  negative SalesAmount: 0
  negative TotalProductCost: 0
  negative Freight: 0
  ShipDate before OrderDate: 0


## 3. Transform

The brief specifically calls out three cleaning steps: standardizing product names,
converting currencies, and categorizing sales transactions - plus adding an Age column
to the customer dimension for the Power BI age-group analysis.


In [10]:
# replace the literal "NA" text with a real missing value across every table,
# so later steps (and SQL) treat it consistently instead of as a real category
for name, df in tables.items():
    tables[name] = df.replace("NA", np.nan)

dim_customer = tables["dim_customer"]
dim_employee = tables["dim_employee"]
dim_geography = tables["dim_geography"]
dim_product = tables["dim_product"]
dim_reseller = tables["dim_reseller"]
dim_sales_territory = tables["dim_sales_territory"]
fact_internet_sales = tables["fact_internet_sales"]
fact_reseller_sales = tables["fact_reseller_sales"]


In [11]:
# standardize product names - trim stray whitespace and make the casing consistent,
# so "ML Fork " and "ml fork" (if they ever show up) are not treated as different products
dim_product["Product"] = dim_product["Product"].str.strip()
dim_product["Model"] = dim_product["Model"].str.strip()
dim_product["Category"] = dim_product["Category"].str.strip().str.title()
dim_product["Subcategory"] = dim_product["Subcategory"].str.strip().str.title()


In [12]:
# currency conversion - every amount in this dataset is already in USD, so there is
# nothing to actually convert. This function is here so the step exists and is easy to
# extend if a future file adds a Currency column with mixed currencies.
def convert_to_usd(amount, currency="USD"):
    rates_to_usd = {"USD": 1.0}   # add more currencies here if the data ever needs it
    return amount * rates_to_usd.get(currency, 1.0)

fact_internet_sales["SalesAmount"] = convert_to_usd(fact_internet_sales["SalesAmount"])
fact_reseller_sales["SalesAmount"] = convert_to_usd(fact_reseller_sales["SalesAmount"])


In [13]:
# categorize each sale by channel, since the two fact tables are the two sales channels
# from the problem statement (direct online retail vs B2B reseller)
fact_internet_sales["Sales_Channel"] = "Online Retail"
fact_reseller_sales["Sales_Channel"] = "B2B Reseller"


In [14]:
# add the Age column the brief asks for, calculated from BirthDate as of the most
# recent order date in the data (not today's real-world date, since this is historical
# sample data - using "today" would make everyone look decades older than intended)
reference_date = pd.to_datetime(fact_internet_sales["OrderDate"]).max()

dim_customer["BirthDate"] = pd.to_datetime(dim_customer["BirthDate"])
dim_customer["Age"] = ((reference_date - dim_customer["BirthDate"]).dt.days // 365).astype(int)

# age bands matching what the brief asks for (5-year bands, under 50 is the group of interest)
bins = [0, 25, 30, 35, 40, 45, 50, 150]
labels = ["Under 25", "25-30", "30-35", "35-40", "40-45", "45-50", "50+"]
dim_customer["Age_Group"] = pd.cut(dim_customer["Age"], bins=bins, labels=labels, right=False)

dim_customer[["CustomerKey", "BirthDate", "Age", "Age_Group"]].head()

,CustomerKey,BirthDate,Age,Age_Group
0,11000,1971-10-06,42,40-45
1,11001,1976-05-10,37,35-40
2,11002,1971-02-09,42,40-45
3,11003,1973-08-14,40,40-45
4,11004,1979-08-05,34,30-35


## 4. Validate (after cleaning)


In [15]:
# row counts should be unchanged - cleaning shouldn't have dropped anyone
for name, df in [("dim_customer", dim_customer), ("dim_product", dim_product),
                  ("fact_internet_sales", fact_internet_sales), ("fact_reseller_sales", fact_reseller_sales)]:
    print(f"{name}: {len(df)} rows")

# Age should be a sensible number for every customer, no negatives or absurd values
print("\nAge range:", dim_customer["Age"].min(), "-", dim_customer["Age"].max())
assert dim_customer["Age"].min() >= 0, "found a negative age - check the reference date"


dim_customer: 18484 rows
dim_product: 397 rows
fact_internet_sales: 60398 rows
fact_reseller_sales: 60899 rows

Age range: 27 - 98


## 5. Load into the MySQL data warehouse

Writing straight from pandas with `to_sql()` is the Python-driven ETL load step the brief
asks for. Note that `to_sql()` does not create primary/foreign keys on its own - those are
added afterward in the SQL script, once the tables actually exist.


In [ ]:
import urllib.parse
user = "root"
password = urllib.parse.quote_plus("Your_Pass")
host = "localhost"
database = "retail_sales_db"

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

tables_to_load = {
    "dim_customer": dim_customer,
    "dim_employee": dim_employee,
    "dim_geography": dim_geography,
    "dim_product": dim_product,
    "dim_reseller": dim_reseller,
    "dim_sales_territory": dim_sales_territory,
    "fact_internet_sales": fact_internet_sales,
    "fact_reseller_sales": fact_reseller_sales,
}

for table_name, df in tables_to_load.items():
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=1000)
    print(f"loaded {table_name}: {len(df)} rows")


loaded dim_customer: 18484 rows
loaded dim_employee: 21 rows
loaded dim_geography: 655 rows
loaded dim_product: 397 rows
loaded dim_reseller: 701 rows
loaded dim_sales_territory: 11 rows
loaded fact_internet_sales: 60398 rows
loaded fact_reseller_sales: 60899 rows


In [17]:
# quick check that everything actually landed in MySQL correctly
for table_name in tables_to_load:
    count = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table_name}", engine)
    print(table_name, "->", count["row_count"].iloc[0], "rows in MySQL")


dim_customer -> 18484 rows in MySQL
dim_employee -> 21 rows in MySQL
dim_geography -> 655 rows in MySQL
dim_product -> 397 rows in MySQL
dim_reseller -> 701 rows in MySQL
dim_sales_territory -> 11 rows in MySQL
fact_internet_sales -> 60398 rows in MySQL
fact_reseller_sales -> 60899 rows in MySQL


In [18]:
dim_customer.to_csv("clean_dim_customer.csv", index=False)
dim_product.to_csv("clean_dim_product.csv", index=False)
dim_employee.to_csv("clean_dim_employee.csv", index=False)
dim_reseller.to_csv("clean_dim_reseller.csv", index=False)
dim_geography.to_csv("clean_dim_geography.csv", index=False)
dim_sales_territory.to_csv("clean_dim_sales_territory.csv", index=False)
fact_internet_sales.to_csv("clean_fact_internet_sales.csv", index=False)
fact_reseller_sales.to_csv("clean_fact_reseller_sales.csv", index=False)
